<a href="https://www.kaggle.com/code/shivanshujaiswara/emotiondetector-ml?scriptVersionId=330870398" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
#Loading necessary libraries
from keras.models import Sequential
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img
from keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from PIL import Image

2026-06-27 20:18:10.116301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782591490.371298      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782591490.438350      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
train_dir = '/kaggle/input/emotiond/train'
test_dir = '/kaggle/input/emotiond/test'

In [3]:
# Function to create a DataFrame with image paths and labels
def createdf(dir):
    img_path = []
    labels = []
    for label in os.listdir(dir):
        for imgname in os.listdir(os.path.join(dir, label)):
            img_path.append(os.path.join(dir, label, imgname))
            labels.append(label)
        print(label, "completed")
    return img_path, labels

#
from tqdm.notebook import tqdm
def extractfeatures(image_path):
    features = []
    for image in tqdm(image_path):
        img = load_img(image, color_mode='grayscale')
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features),48,48,1)
    return features

In [4]:
#Creating DataFrame for training data
train = pd.DataFrame()
train['img_path'], train['label'] = createdf(train_dir)

#Creating DataFrame for test data
test = pd.DataFrame()
test['img_path'], test['label'] = createdf(test_dir)

fearful completed
disgusted completed
angry completed
neutral completed
sad completed
surprised completed
happy completed
fearful completed
disgusted completed
angry completed
neutral completed
sad completed
surprised completed
happy completed


In [5]:
train_features = extractfeatures(train['img_path'])
test_features = extractfeatures(test['img_path'])

  0%|          | 0/28709 [00:00<?, ?it/s]

  0%|          | 0/7178 [00:00<?, ?it/s]

In [6]:
x_train = train_features/255.0
x_test = test_features/255.0

le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [7]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [8]:
#Making the model
model = Sequential()

#convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

#fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

#output layer
model.add(Dense(7, activation='softmax'))

#compiling Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1782591648.770275      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782591648.773034      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [9]:
model.fit(x= x_train,y = y_train, epochs=100, batch_size=128, validation_data=(x_test, y_test))

Epoch 1/100


I0000 00:00:1782591654.191405      61 service.cc:148] XLA service 0x79df0400f1a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782591654.192289      61 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782591654.192311      61 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782591654.615532      61 cuda_dnn.cc:529] Loaded cuDNN version 90300


  3/225 ━━━━━━━━━━━━━━━━━━━━ 13s 62ms/step - accuracy: 0.2044 - loss: 1.9757

I0000 00:00:1782591662.630829      61 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


225/225 ━━━━━━━━━━━━━━━━━━━━ 29s 78ms/step - accuracy: 0.2302 - loss: 1.8548 - val_accuracy: 0.2810 - val_loss: 1.7531
Epoch 2/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.3068 - loss: 1.7080 - val_accuracy: 0.4143 - val_loss: 1.4993
Epoch 3/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.4132 - loss: 1.5193 - val_accuracy: 0.4774 - val_loss: 1.3740
Epoch 4/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.4571 - loss: 1.4103 - val_accuracy: 0.4813 - val_loss: 1.3287
Epoch 5/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.4837 - loss: 1.3470 - val_accuracy: 0.5234 - val_loss: 1.2298
Epoch 6/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.5056 - loss: 1.2932 - val_accuracy: 0.5358 - val_loss: 1.2164
Epoch 7/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.5255 - loss: 1.2562 - val_accuracy: 0.5552 - val_loss: 1.1668
Epoch 8/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.5428 - loss: 1.2053 - val_

In [10]:
model_json = model.to_json()
with open("emotion_detector.json", "w") as json_file:
    json_file.write(model_json)
model.save("emotion_detector.h5")